# LAB 04 — IMDB: Dataset inspection, split chống leakage và baseline phân loại
Notebook Colab này thực hiện các mục 11, 12, 13 của đề. Dữ liệu được gộp thành một tập trước khi chia phân tầng theo nhãn thành train/validation/test = 60%/20%/20%. Với `imdb.zip` hiện có, ZIP chứa một CSV `IMDB Dataset.csv` với hai cột `review`, `sentiment` (50.000 review); nó không chứa hai thư mục train/test riêng. Nếu ZIP của bạn có hai split, cell đọc dữ liệu bên dưới hỗ trợ CSV có tên train/test hoặc cấu trúc thư mục `train|test/positive|negative/*.txt`.

> Nếu dùng Google Drive: sửa `ZIP_PATH` thành đường dẫn chính xác trong Shared with me (hoặc tạo shortcut vào MyDrive). Colab thường không đọc trực tiếp Shared with me cho đến khi file/shortcut xuất hiện trong MyDrive. Có thể tải `imdb.zip` lên phiên Colab và giữ mặc định `/content/imdb.zip`.

In [2]:
# Chạy trên Google Colab
from pathlib import Path
import zipfile, csv, html, re, os
import numpy as np
import pandas as pd

ZIP_PATH = Path('/content/imdb.zip')

if not ZIP_PATH.exists():
    from google.colab import files
    print('Chưa tìm thấy ZIP_PATH. Hãy tải imdb.zip lên Colab.')
    uploaded = files.upload()
    ZIP_PATH = Path(next(iter(uploaded)))
print('ZIP:', ZIP_PATH, '| size:', f'{ZIP_PATH.stat().st_size/1024/1024:.2f} MB')

Mounted at /content/drive
Chưa tìm thấy ZIP_PATH. Hãy tải imdb.zip lên Colab.


Saving imdb.zip to imdb.zip
ZIP: imdb.zip | size: 25.71 MB


## 11. Experiment 1 — Dataset inspection
Đọc cả train và test nếu ZIP thực sự có sẵn hai split, sau đó ghép lại. Nếu ZIP chứa một CSV tổng hợp như file đang có, đọc CSV đó. Với thư mục IMDB chuẩn, nhãn được lấy từ tên thư mục positive/negative.

In [3]:
def load_imdb_zip(zip_path):
    frames = []
    with zipfile.ZipFile(zip_path) as z:
        names = [n for n in z.namelist() if not n.endswith('/')]
        # Dạng thư mục IMDB chuẩn: train/test/pos|neg hoặc positive|negative/*.txt
        txts = [n for n in names if n.lower().endswith('.txt')]
        for name in txts:
            parts = name.lower().replace('\\', '/').split('/')
            label = next((p for p in parts if p in ('pos','positive','neg','negative')), None)
            if label is not None:
                y = 'positive' if label in ('pos','positive') else 'negative'
                frames.append({'review': z.read(name).decode('utf-8', errors='replace'), 'sentiment': y, 'source_split': next((p for p in parts if p in ('train','test')), 'unspecified')})
        if frames:
            return pd.DataFrame(frames)
        # Dạng CSV: có thể là một CSV tổng hợp hoặc các train/test CSV riêng
        csvs = [n for n in names if n.lower().endswith('.csv')]
        if not csvs:
            raise ValueError('Không tìm thấy .csv hoặc các file .txt nằm trong thư mục pos/neg của ZIP.')
        for name in csvs:
            with z.open(name) as f:
                df = pd.read_csv(f)
            df.columns = [str(c).strip().lower() for c in df.columns]
            if {'review','sentiment'}.issubset(df.columns):
                part = df[['review','sentiment']].copy()
                part['source_split'] = 'train' if 'train' in Path(name).name.lower() else ('test' if 'test' in Path(name).name.lower() else 'single_csv')
            elif {'text','label'}.issubset(df.columns):
                part = df.rename(columns={'text':'review','label':'sentiment'})[['review','sentiment']].copy()
                part['source_split'] = 'train' if 'train' in Path(name).name.lower() else ('test' if 'test' in Path(name).name.lower() else 'single_csv')
            else:
                raise ValueError(f'{name}: cần cột review/sentiment hoặc text/label; thấy {list(df.columns)}')
            frames.append(part)
    return pd.concat(frames, ignore_index=True)

df = load_imdb_zip(ZIP_PATH)
df['review'] = df['review'].fillna('').astype(str)
df['sentiment'] = df['sentiment'].astype(str).str.strip().str.lower()
df = df[df['sentiment'].isin(['positive','negative'])].copy()
df['label'] = df['sentiment'].map({'negative':0, 'positive':1})
print('Số mẫu sau khi ghép/chuẩn hóa:', len(df))
print('Nguồn split trong ZIP (chỉ để kiểm tra):')
print(df['source_split'].value_counts(dropna=False).to_string())
print('\nNhãn:', sorted(df['sentiment'].unique()))
print('Thiếu review:', int(df['review'].isna().sum()), '| review rỗng:', int(df['review'].str.strip().eq('').sum()))
print('Trùng review:', int(df['review'].duplicated().sum()))
df['n_words'] = df['review'].str.split().str.len()
print('\nPhân bố lớp:')
display(pd.concat([df['sentiment'].value_counts().rename('count'), (df['sentiment'].value_counts(normalize=True)*100).round(2).rename('percent')], axis=1))
print('Độ dài review (số từ):')
display(df['n_words'].describe().round(2))
display(df[['review','sentiment','source_split','n_words']].head(3))

Số mẫu sau khi ghép/chuẩn hóa: 50000
Nguồn split trong ZIP (chỉ để kiểm tra):
source_split
single_csv    50000

Nhãn: ['negative', 'positive']
Thiếu review: 0 | review rỗng: 0
Trùng review: 418

Phân bố lớp:


,count,percent
sentiment,,
positive,25000,50.0
negative,25000,50.0


Độ dài review (số từ):


,n_words
count,50000.00
mean,231.16
std,171.34
min,4.00
25%,126.00
50%,173.00
75%,280.00
max,2470.00


,review,sentiment,source_split,n_words
0,One of the other reviewers has mentioned that ...,positive,single_csv,307
1,A wonderful little production. <br /><br />The...,positive,single_csv,162
2,I thought this was a wonderful way to spend ti...,positive,single_csv,166


### Trả lời câu hỏi mục 11
Bộ IMDB là bài toán **phân loại cảm xúc nhị phân**: positive và negative. Bảng phân bố lớp ở output cho biết dữ liệu có mất cân bằng hay không; tỷ lệ hai lớp gần 50/50 thì tương đối cân bằng. Các kiểm tra số mẫu, độ dài, review rỗng/trùng và nhãn giúp phát hiện dữ liệu bất thường trước khi huấn luyện. Với ZIP hiện tại là CSV tổng hợp, không có split train/test bên trong để cộng; ta dùng toàn bộ 50.000 mẫu rồi tạo split mới. Nếu ZIP khác có train và test, loader gộp hai nguồn trước khi chia lại.

## 12. Data leakage — chia dữ liệu trước, fit TF-IDF sau
Dùng chia phân tầng để mỗi tập giữ gần tỷ lệ positive/negative như toàn bộ dữ liệu. `TfidfVectorizer` nằm bên trong `Pipeline`; khi gọi `fit` chỉ trên train, vocabulary và IDF chỉ được học từ train. Validation/test chỉ được `transform`, không tham gia học đặc trưng. Nếu có review trùng lặp, loại trùng trước khi chia để tránh cùng một review lọt vào nhiều tập.

In [4]:
from sklearn.model_selection import train_test_split

# Loại bỏ review trùng trước khi split để cùng văn bản không rơi vào các tập khác nhau.
before = len(df)
df = df.drop_duplicates(subset='review').reset_index(drop=True)
print('Đã loại review trùng:', before - len(df))
X = df['review']
y = df['label']

# 20% test; 80% còn lại tách thành 60% train và 20% validation.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

splits = {'train': y_train, 'validation': y_val, 'test': y_test}
print('\nKích thước và phân bố nhãn từng tập:')
for name, labels in splits.items():
    print(f'\n{name}: {len(labels):,} mẫu ({len(labels)/len(df):.1%})')
    print(labels.map({0:'negative',1:'positive'}).value_counts(normalize=True).mul(100).round(2).astype(str).add('%').to_string())
assert len(X_train)+len(X_val)+len(X_test) == len(df)
assert set(X_train.index).isdisjoint(X_val.index) and set(X_train.index).isdisjoint(X_test.index) and set(X_val.index).isdisjoint(X_test.index)
print('\nKiểm tra: các split không chồng lấp và tổng số mẫu khớp.')

Đã loại review trùng: 418

Kích thước và phân bố nhãn từng tập:

train: 29,748 mẫu (60.0%)
label
positive    50.19%
negative    49.81%

validation: 9,917 mẫu (20.0%)
label
positive    50.19%
negative    49.81%

test: 9,917 mẫu (20.0%)
label
positive    50.19%
negative    49.81%

Kiểm tra: các split không chồng lấp và tổng số mẫu khớp.


### Trả lời câu hỏi mục 12
Nếu fit TF-IDF trên toàn bộ dữ liệu rồi mới chia, vocabulary và IDF đã dùng thông tin từ validation/test. Mô hình vì vậy được tiếp xúc với phân bố của dữ liệu đánh giá trước khi kiểm tra, làm kết quả đánh giá kém khách quan (data leakage). Quy trình đúng là chia trước; fit vectorizer và classifier trên train; chỉ dùng validation để lựa chọn thiết lập; test giữ đến bước đánh giá cuối cùng. Trong notebook này, pipeline được fit trên train và test chỉ được chạm tới ở mục đánh giá.

## 13. Core implementation — TF-IDF + Logistic Regression
Pipeline thực hiện văn bản thô → TF-IDF → Logistic Regression → dự đoán. Không bỏ stopwords để tránh xóa nhầm từ phủ định như `not`. Validation được dùng để xem kết quả ban đầu; test dùng một lần cho báo cáo cuối.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1,1), min_df=2,
                              strip_accents='unicode', sublinear_tf=True)),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
model.fit(X_train, y_train)
print('Số feature TF-IDF học từ train:', len(model.named_steps['tfidf'].vocabulary_))

def report_split(name, X_part, y_part):
    pred = model.predict(X_part)
    print(f'\n===== {name} =====')
    print(f'Accuracy : {accuracy_score(y_part,pred):.4f}')
    print(f'Precision: {precision_score(y_part,pred,zero_division=0):.4f} (positive)')
    print(f'Recall   : {recall_score(y_part,pred,zero_division=0):.4f} (positive)')
    print(f'F1       : {f1_score(y_part,pred,zero_division=0):.4f} (positive)')
    print('Confusion matrix [ [TN, FP], [FN, TP] ]:')
    print(confusion_matrix(y_part, pred, labels=[0,1]))
    print(classification_report(y_part, pred, target_names=['negative','positive'], digits=4, zero_division=0))
    return pred

val_pred = report_split('VALIDATION (dùng để kiểm tra/chọn model)', X_val, y_val)
test_pred = report_split('TEST (đánh giá cuối cùng)', X_test, y_test)

Số feature TF-IDF học từ train: 48487

===== VALIDATION (dùng để kiểm tra/chọn model) =====
Accuracy : 0.8936
Precision: 0.8850 (positive)
Recall   : 0.9058 (positive)
F1       : 0.8952 (positive)
Confusion matrix [ [TN, FP], [FN, TP] ]:
[[4354  586]
 [ 469 4508]]
              precision    recall  f1-score   support

    negative     0.9028    0.8814    0.8919      4940
    positive     0.8850    0.9058    0.8952      4977

    accuracy                         0.8936      9917
   macro avg     0.8939    0.8936    0.8936      9917
weighted avg     0.8938    0.8936    0.8936      9917


===== TEST (đánh giá cuối cùng) =====
Accuracy : 0.8966
Precision: 0.8868 (positive)
Recall   : 0.9102 (positive)
F1       : 0.8984 (positive)
Confusion matrix [ [TN, FP], [FN, TP] ]:
[[4362  578]
 [ 447 4530]]
              precision    recall  f1-score   support

    negative     0.9070    0.8830    0.8949      4940
    positive     0.8868    0.9102    0.8984      4977

    accuracy                    

### Trả lời câu hỏi mục 13
Đây là baseline supervised learning: mỗi review được biến thành vector TF-IDF, sau đó Logistic Regression học trọng số để dự đoán positive/negative. `model.fit(X_train, y_train)` chỉ học trên train. Output gồm accuracy, precision, recall, F1, confusion matrix và precision/recall/F1 theo từng lớp; nên xem các chỉ số này cùng nhau thay vì chỉ nhìn accuracy. Nếu số lượng mỗi lớp gần nhau, accuracy hữu ích hơn; nếu mất cân bằng, F1/recall của lớp cần quan tâm và confusion matrix quan trọng hơn. Kết quả cụ thể sẽ được in khi chạy notebook trên Colab.